<a href="https://colab.research.google.com/github/OutForMilks/Hunger/blob/main/g2p.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Setup**

**Google Collab**

In [6]:
# !git clone https://github.com/OutForMilks/Hunger.git
# %cd /content/Hunger

In [1]:
import pandas as pd
import os
from pathlib import Path
import glob

import torch
import torch.nn as nn
from torch.utils.data import WeightedRandomSampler
import sys

from src.prep_data import *
from src.decode import *
from src.train import *
from src.temperature import *
from src.evaluation import *

# **Prepare Data**

In [2]:
col_names = ["text", "phonetic", "language", "group"]

### Philippine Languages

In [3]:
ph_files = glob.glob("wikipron/filipino/*.tsv")
ph_languages = ["bikolano", "cebuano", "hiligaynon", "ilocano", "kapampangan", "pangasinan", "tagalog", "waray"]

temp_ph = []
for filename in ph_files:
    temp_df = pd.read_csv(filename, sep='\t', names=col_names, header=None)
    temp_df["language"] = Path(filename).stem
    temp_df["group"] = "philippine"
    temp_ph.append(temp_df)

ph_df = pd.concat(temp_ph, ignore_index=True)

print(ph_df.head())
print("\n")
print(ph_df["language"].unique())

      text       phonetic  language       group
0    Abril    ʔ a b ɾ i l  bikolano  philippine
1   Agosto  ʔ a ɡ o s t o  bikolano  philippine
2  Aguilar  ʔ a ɡ i l a ɾ  bikolano  philippine
3    Albay    ʔ a l b a j  bikolano  philippine
4   Alzaga  ʔ a l s a ɡ a  bikolano  philippine


<StringArray>
[   'bikolano',     'cebuano',  'hiligaynon',     'ilocano', 'kapampangan',
  'pangasinan',       'waray']
Length: 7, dtype: str


### Tagalog

In [4]:
input_dir_tgl = Path("wikipron/tagalog_cleaned.tsv")

df_tgl = pd.read_csv(input_dir_tgl, sep='\t', names=col_names, header=None)
df_tgl["language"] = "tagalog"
df_tgl["group"] = "philippine"

print(df_tgl.head())

   text phonetic language       group
0     g        ŋ  tagalog  philippine
1     m        m  tagalog  philippine
2    ng        ŋ  tagalog  philippine
3  'day    d a j  tagalog  philippine
4   'di    d i ʔ  tagalog  philippine


### Spanish

In [5]:
input_dir_spa = Path("wikipron/castillian_spanish_filtered.tsv")

df_spa = pd.read_csv(input_dir_spa, sep='\t', names=col_names, header=None)
df_spa["language"] = "spanish"
df_spa["group"] = "spanish"

print(df_spa.head())

     text   phonetic language    group
0  'tamo'    t a m o  spanish  spanish
1  'tamos  t a m o s  spanish  spanish
2    'tas      t a s  spanish  spanish
3    ACIP    a θ i p  spanish  spanish
4   ACNUR  a ɡ n u ɾ  spanish  spanish


### Splitting Tagalog

In [6]:
from sklearn.model_selection import train_test_split

# 70/15/15 split: peel off 15% for test, then 17.65% of the remaining 85% (~15% overall) for dev
train_dev_tgl, test_tgl = train_test_split(df_tgl, test_size=0.15, random_state=42)
train_tgl, dev_tgl = train_test_split(train_dev_tgl, test_size=0.1765, random_state=42)

ph_aux_df = ph_df.copy()
spa_aux_df = df_spa.copy()

# Each training mix = the tagalog train split + its auxiliary language group
tagalog_only_df = train_tgl.copy()
ph_df = pd.concat([train_tgl, ph_aux_df, spa_aux_df], ignore_index=True)

# Clean only now, so duplicates between the tagalog split and the aux data are caught too
for d in [tagalog_only_df, ph_df, df_spa]:
    d.drop_duplicates(subset=["text", "phonetic"], inplace=True)
    d.dropna(inplace=True)

In [7]:
input_dir = Path("data/input")
input_dir.mkdir(parents=True, exist_ok=True)

output_dir = Path("data/splits")
output_dir.mkdir(parents=True, exist_ok=True)

# Pool all training data, dedupe, then write one {lang}_train.tsv per language
# so convert_split can assemble any subset of languages into a training set.
combined_df = pd.concat(
    [ph_aux_df, spa_aux_df, train_tgl], ignore_index=True
).drop_duplicates(subset=["text", "phonetic"])

for lang, group_df in combined_df.groupby("language"):
    group_df[["text", "phonetic"]].to_csv(
        input_dir / f"{lang}_train.tsv", sep="\t", index=False, header=False
    )
    print(f"{lang}: {len(group_df)} pairs")

# The held-out tagalog dev/test splits -- every model is evaluated on these
dev_tgl[["text", "phonetic"]].to_csv(input_dir / "tagalog_dev.tsv", sep="\t", index=False, header=False)
test_tgl[["text", "phonetic"]].to_csv(input_dir / "tagalog_test.tsv", sep="\t", index=False, header=False)

bikolano: 5432 pairs
cebuano: 3031 pairs
hiligaynon: 165 pairs
ilocano: 827 pairs
kapampangan: 900 pairs
pangasinan: 158 pairs
spanish: 131869 pairs
tagalog: 13365 pairs
waray: 186 pairs


In [8]:
sets = {
    "ph_spanish": ph_languages + ["spanish"],
}

for set_name, langs in sets.items():
    out_dir = f"data/splits/{set_name}"
    print(f"Set: {set_name}")
    convert_split(input_dir, out_dir, langs, "train")

for split in ["dev", "test"]:
    out_dir = f"data/splits/tgl_{split}"
    print(f"Split: {split}")
    convert_split(input_dir, out_dir, ["tagalog"], split)

Set: ph_spanish
E:\Media-Files\Real-Stuff\School\Code-Files\Cloned-Repositories\Salo-Salo
  bikolano train: 5432 pairs
  cebuano train: 3031 pairs
  hiligaynon train: 165 pairs
  ilocano train: 827 pairs
  kapampangan train: 900 pairs
  pangasinan train: 158 pairs
  tagalog train: 13365 pairs
  waray train: 186 pairs
  spanish train: 131869 pairs
  -> wrote 155933 lines to train.{src,tgt,lang}
Split: dev
E:\Media-Files\Real-Stuff\School\Code-Files\Cloned-Repositories\Salo-Salo
  tagalog dev: 3259 pairs
  -> wrote 3259 lines to dev.{src,tgt,lang}
Split: test
E:\Media-Files\Real-Stuff\School\Code-Files\Cloned-Repositories\Salo-Salo
  tagalog test: 3259 pairs
  -> wrote 3259 lines to test.{src,tgt,lang}


# **Train**

In [15]:
DATA = Path("data/splits")
OUTPUT = Path("models")
#seed is a var
STEPS = 200000
SAVE_STEP = 50000

BATCH_SIZE = 256
HIDDEN_SIZE = 512
FF_SIZE = 2048
N_LAYERS = 6
NUM_HEADS = 8

DROPOUT = 0.1
WARMUP = 4000
SMOOTHING = 0.1

# if len(xm.get_xla_supported_devices()) > 0:
#     DEVICE = "xla"
if torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

xla = DEVICE == "xla"

print(DEVICE)

#loging is a var

CONFIG = {
    "data_path": str(DATA),
    "output_path": str(OUTPUT),

    "steps": STEPS,
    "save_step": SAVE_STEP,

    "batch_size": BATCH_SIZE,
    "hidden_size": HIDDEN_SIZE,
    "ff_size": FF_SIZE,
    "n_layers": N_LAYERS,
    "num_heads": NUM_HEADS,

    "dropout": DROPOUT,
    "warmup": WARMUP,
    "label_smoothing": SMOOTHING,

    "device": DEVICE,
}

log_step = 1000

cuda


In [16]:
seeds = [42, 67, 888]
seed_checkpoint_results = {}

for seed in seeds:
    device = torch.device(DEVICE)
    torch.manual_seed(seed)

    # bf16 autocast on CUDA: tensor-core matmuls + roughly halved activation
    # memory; unlike fp16 it needs no GradScaler (same exponent range as fp32).
    use_amp = DEVICE == "cuda" and torch.cuda.is_bf16_supported()

    print(f"\nSeed: {seed}")
    DATA = Path(f"data/splits")
    OUTPUT = Path(f"models/seed_{seed}")

    os.makedirs(OUTPUT, exist_ok=True)
    
    expected_steps = list(range(SAVE_STEP, STEPS + 1, SAVE_STEP))
    expected_ckpts = [os.path.join(OUTPUT, f"ckpt_seed{seed}_checkpoint{s}.pt") for s in expected_steps]
    all_exist = all(os.path.exists(p) for p in expected_ckpts)


    if not all_exist:
        config_1 = dict(CONFIG)
        config_1["seed"] = seed
        config_1["data_path"] = str(DATA)
        config_1["output_path"] = str(OUTPUT)
        config_1["temperature"] = 3.0 

        # Re-seed per run so each model trains from the same init regardless of set order
        torch.manual_seed(seed)
        device = torch.device(DEVICE)

        # Vocab is built from this set's training data only (each set has its own
        # grapheme/phoneme inventory), then saved alongside the checkpoints.
        src_vocab, tgt_vocab = build_vocabs(
            os.path.join(DATA, "train.src"), os.path.join(DATA, "train.tgt"))
        save_vocabs(src_vocab, tgt_vocab, os.path.join(OUTPUT, "vocab.json"))

        train_ds = G2PDataset(os.path.join(DATA, "train.src"),
                                os.path.join(DATA, "train.tgt"),
                                src_vocab, tgt_vocab)
        # Fixed max lengths keep every batch the same shape -- required on XLA.
        # On GPU, pad each batch only to its own max instead: mean word length is
        # ~8 graphemes vs. a global max of ~48, so this skips most padding compute.
        max_src, max_tgt = compute_max_lengths(train_ds)
        print(f"src vocab={len(src_vocab)} tgt vocab={len(tgt_vocab)} "
                f"examples={len(train_ds)} | fixed shapes: src={max_src} tgt={max_tgt}")

        # Temperature sampling (T=5) flattens the language imbalance: low-resource
        # languages get sampled far above their natural frequency.
        weights = temperature_sampling_weights(
            os.path.join(DATA, "train.lang"), temperature=3.0
        )
        sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

        collate_fn = (make_fixed_collate(max_src, max_tgt) if xla
                    else make_dynamic_collate())
        loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                            collate_fn=collate_fn,
                            drop_last=True)

        model = Transformer(len(src_vocab), len(tgt_vocab), d_model=HIDDEN_SIZE,
                            n_heads=NUM_HEADS, d_ff=FF_SIZE,
                            n_layers=N_LAYERS, dropout=DROPOUT,
                            pad_id=PAD_ID).to(device)
        # lr=0 because NoamLR fully controls the learning rate (warmup then decay)
        opt = torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
        sched = NoamLR(opt, HIDDEN_SIZE, WARMUP, xla=xla)
        criterion = LabelSmoothingLoss(len(tgt_vocab), PAD_ID, SMOOTHING).to(device)

        model.train()
        data_iter = infinite_loader(loader)
        for step in range(1, STEPS + 1):
            batch = next(data_iter)
            if xla:
                src, tgt_in, tgt_out, src_pad, tgt_pad = batch  # already on device
            else:
                src, tgt_in, tgt_out, src_pad, tgt_pad = (x.to(device) for x in batch)

            # Teacher forcing: tgt_in is the shifted target, tgt_out is what we score against
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=use_amp):
                logits = model(src, tgt_in, src_pad, tgt_pad)
                loss = criterion(logits, tgt_out)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            sched.step()  # calls xm.optimizer_step on XLA -> executes the graph

            if step % log_step == 0:
                # .item() forces a host sync; only do it at log intervals on TPU.
                print(f"step {step:>7}/{STEPS}  loss {loss.item():.4f}  "
                        f"lr {opt.param_groups[0]['lr']:.2e}")

            if step % SAVE_STEP == 0 or step == STEPS:
                # Checkpoint bundles weights + config + both vocabs, so evaluation
                # can rebuild the model from the .pt file alone.
                ckpt_path = os.path.join(OUTPUT, f"ckpt_seed{seed}_checkpoint{step}.pt")
                payload = {"model": model.state_dict(),
                            "config": config_1,
                            "src_vocab": src_vocab.to_dict(),
                            "tgt_vocab": tgt_vocab.to_dict()}
                torch.save(payload, ckpt_path)
                print(f"  saved {ckpt_path}")

        del model, opt, sched, criterion, loader, train_ds
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    else:
        print(f"\nSkipping seed {seed}, all checkpoints already exist")
    
    for step in expected_steps:
        ckpt_path = os.path.join(OUTPUT, f"ckpt_seed{seed}_checkpoint{step}.pt")
        per, wer, _, _ = batch_evaluate_checkpoint_beam(
            model_paths=[ckpt_path],
            src_path="./data/splits/tgl_dev/dev.src",
            tgt_path="./data/splits/tgl_dev/dev.tgt",
            device=device,
            batch_size=64,
        )
        seed_checkpoint_results[(seed, step)] = {"PER": per, "WER": wer}
        print(f"Seed={seed}, Checkpoint={step}: dev PER={per:.4f}  WER={wer:.4f}")


Seed: 42

Skipping seed 42, all checkpoints already exist
  decoded 64/3259
  decoded 128/3259
  decoded 192/3259
  decoded 256/3259
  decoded 320/3259
  decoded 384/3259
  decoded 448/3259
  decoded 512/3259
  decoded 576/3259
  decoded 640/3259
  decoded 704/3259
  decoded 768/3259
  decoded 832/3259
  decoded 896/3259
  decoded 960/3259
  decoded 1024/3259
  decoded 1088/3259
  decoded 1152/3259
  decoded 1216/3259
  decoded 1280/3259
  decoded 1344/3259
  decoded 1408/3259
  decoded 1472/3259
  decoded 1536/3259
  decoded 1600/3259
  decoded 1664/3259
  decoded 1728/3259
  decoded 1792/3259
  decoded 1856/3259
  decoded 1920/3259
  decoded 1984/3259
  decoded 2048/3259
  decoded 2112/3259
  decoded 2176/3259
  decoded 2240/3259
  decoded 2304/3259
  decoded 2368/3259
  decoded 2432/3259
  decoded 2496/3259
  decoded 2560/3259
  decoded 2624/3259
  decoded 2688/3259
  decoded 2752/3259
  decoded 2816/3259
  decoded 2880/3259
  decoded 2944/3259
  decoded 3008/3259
  decoded 3072/32

In [17]:
print("\nSeed per checkpoint summary:")
for (seed, ckpt), r in sorted(seed_checkpoint_results.items(), key=lambda x: x[1]["PER"]):
    print(f"Seed = {seed}, Checkpoint={ckpt}: PER={r['PER']:.4f}  WER={r['WER']:.4f}")

best_key = min(seed_checkpoint_results, key=lambda t: seed_checkpoint_results[t]["PER"])
best_seed, best_ckpt = best_key
print(f"\nBest combination: Seed={best_seed} Checkpoint={best_ckpt}")
print(f"\nPER={seed_checkpoint_results[best_key]['PER']:.4f}, WER={seed_checkpoint_results[best_key]['WER']:.4f})")


Seed per checkpoint summary:
Seed = 888, Checkpoint=200000: PER=0.0167  WER=0.1135
Seed = 42, Checkpoint=150000: PER=0.0167  WER=0.1117
Seed = 42, Checkpoint=100000: PER=0.0170  WER=0.1157
Seed = 888, Checkpoint=150000: PER=0.0174  WER=0.1169
Seed = 42, Checkpoint=50000: PER=0.0188  WER=0.1209
Seed = 888, Checkpoint=100000: PER=0.0192  WER=0.1154
Seed = 888, Checkpoint=50000: PER=0.0198  WER=0.1172
Seed = 67, Checkpoint=100000: PER=0.0203  WER=0.1105
Seed = 42, Checkpoint=200000: PER=0.0208  WER=0.1105
Seed = 67, Checkpoint=50000: PER=0.0229  WER=0.1212
Seed = 67, Checkpoint=200000: PER=0.0232  WER=0.1105
Seed = 67, Checkpoint=150000: PER=0.0237  WER=0.1141

Best combination: Seed=888 Checkpoint=200000

PER=0.0167, WER=0.1135)


# **Decode**

In [10]:
ensemble_paths = []
seeds = [42, 67, 888]

for seed in seeds:
    ensemble_paths.extend(sorted(glob.glob(f"models/5090Attempt/200kattempt/seed_{seed}/ckpt_*.pt")))

print(f"\nEnsemble of {len(ensemble_paths)} checkpoints")
print(ensemble_paths)


Ensemble of 12 checkpoints
['models/5090Attempt/200kattempt/seed_42\\ckpt_seed42_checkpoint100000.pt', 'models/5090Attempt/200kattempt/seed_42\\ckpt_seed42_checkpoint150000.pt', 'models/5090Attempt/200kattempt/seed_42\\ckpt_seed42_checkpoint200000.pt', 'models/5090Attempt/200kattempt/seed_42\\ckpt_seed42_checkpoint50000.pt', 'models/5090Attempt/200kattempt/seed_67\\ckpt_seed67_checkpoint100000.pt', 'models/5090Attempt/200kattempt/seed_67\\ckpt_seed67_checkpoint150000.pt', 'models/5090Attempt/200kattempt/seed_67\\ckpt_seed67_checkpoint200000.pt', 'models/5090Attempt/200kattempt/seed_67\\ckpt_seed67_checkpoint50000.pt', 'models/5090Attempt/200kattempt/seed_888\\ckpt_seed888_checkpoint100000.pt', 'models/5090Attempt/200kattempt/seed_888\\ckpt_seed888_checkpoint150000.pt', 'models/5090Attempt/200kattempt/seed_888\\ckpt_seed888_checkpoint200000.pt', 'models/5090Attempt/200kattempt/seed_888\\ckpt_seed888_checkpoint50000.pt']


In [19]:
per, wer, refs, hyps = batch_evaluate_checkpoint_beam(
    model_paths=ensemble_paths,
    src_path="data/splits/tgl_test/test.src",
    tgt_path="data/splits/tgl_test/test.tgt",
    device=device,
    batch_size=64,
    beam=5,
)
print(f"\nEnsemble ({len(ensemble_paths)} models): PER={per:.4f}  WER={wer:.4f}")

  decoded 64/3259
  decoded 128/3259
  decoded 192/3259
  decoded 256/3259
  decoded 320/3259
  decoded 384/3259
  decoded 448/3259
  decoded 512/3259
  decoded 576/3259
  decoded 640/3259
  decoded 704/3259
  decoded 768/3259
  decoded 832/3259
  decoded 896/3259
  decoded 960/3259
  decoded 1024/3259
  decoded 1088/3259
  decoded 1152/3259
  decoded 1216/3259
  decoded 1280/3259
  decoded 1344/3259
  decoded 1408/3259
  decoded 1472/3259
  decoded 1536/3259
  decoded 1600/3259
  decoded 1664/3259
  decoded 1728/3259
  decoded 1792/3259
  decoded 1856/3259
  decoded 1920/3259
  decoded 1984/3259
  decoded 2048/3259
  decoded 2112/3259
  decoded 2176/3259
  decoded 2240/3259
  decoded 2304/3259
  decoded 2368/3259
  decoded 2432/3259
  decoded 2496/3259
  decoded 2560/3259
  decoded 2624/3259
  decoded 2688/3259
  decoded 2752/3259
  decoded 2816/3259
  decoded 2880/3259
  decoded 2944/3259
  decoded 3008/3259
  decoded 3072/3259
  decoded 3136/3259
  decoded 3200/3259
  decoded 3259/3

# **Evaluation**
The ensemble model is used once on the test-split of 3259 Filipino words. Both PER and WER are used as an evaluation metric following the standard practice of existing literature. The model yielded a PER of 0.0128 (1.28%) and a WER of 0.0871 (8.71%). 

In [9]:
import pandas as pd

df = pd.read_csv("ensemble_12models_test_results.csv")

errors_only = df[df["is_error"] == True]

print(len(errors_only))

# convert to use later
refs_list = [str(x).strip().split() for x in df["target_ref"]]
hyps_list = [str(x).strip().split() for x in df["model_hyp"]]
refs_dict = dict(enumerate(refs_list))
hyps_dict = dict(enumerate(hyps_list))

print(errors_only.sample(10))

284
                             input_word                   target_ref  \
1633  <tagalog> t a t a k k a l a k a l      t a t a k k a l a k a l   
715                   <tagalog> e y t s                    ʔ e j t͡ʃ   
774           <tagalog> b a l i s u s o            b a l i s u s o ʔ   
2047  <tagalog> k a n u n u n u n u a n  k a n u n u ʔ n u n u ʔ a n   
2118        <tagalog> a l u p a k a y a        ʔ a l u p a k a j a ʔ   
2532              <tagalog> k u w a l a                    k u a l a   
2602                  <tagalog> h i r a                      h i ɾ a   
2870            <tagalog> l u n u k i n                l u n u k i n   
2449      <tagalog> a t a n g a t a n g          ʔ a t a ŋ ʔ a t a ŋ   
602               <tagalog> m a i k s i              m a ʔ i k s i ʔ   

                      model_hyp  is_error  
1633      t a t a k a l a k a l      True  
715                   ʔ e j t s      True  
774             b a l i s u s o      True  
2047  k a n u n u n u n u ʔ

For further error analysis beyond PER measurement, errors are classified by their type of error, phoneme unit, and substitution pair.

In [10]:
from collections import defaultdict, Counter

def align_trxn(seqA, seqB):
    """Returns two aligned transcriptions (each a list of phones) and their edit distance given two unaligned transcriptions."""

    lenA = len(seqA)
    lenB = len(seqB)

    score_matrix        = [[0 for _ in range(lenB+1)] for _ in range(lenA+1)]
    traceback_matrix    = [[0 for _ in range(lenB+1)] for _ in range(lenA+1)]

    # define substitution scores and gap penalty
    match       = +1
    mismatch    = -1
    gap         = -1

    # initialize scoring matrix
    for i in range(lenA+1):
        score_matrix[i][0] = i*gap  # rows: seqA
    for j in range(lenB+1):
        score_matrix[0][j] = j*gap  # cols: seqB

    # fill scoring matrix
    for i in range(1, lenA+1):
        for j in range(1, lenB+1):
            if seqA[i-1] == seqB[j-1]:
                score = match
            else:
                score = mismatch

            # final score choices
            aligned     = score_matrix[i-1][j-1] + score
            deleted     = score_matrix[i-1][j  ] + gap
            inserted    = score_matrix[i  ][j-1] + gap

            score_matrix[i][j] = max(aligned, deleted, inserted)

            if score_matrix[i][j] == aligned:
                traceback_matrix[i][j] = 0  # phones i and j aligned
            elif score_matrix[i][j] == deleted:
                traceback_matrix[i][j] = 1  # gap in seqB
            elif score_matrix[i][j] == inserted:
                traceback_matrix[i][j] = -1 # gap in seqA

    # print(score_matrix)

    # get aligned sequences from traceback matrix
    i, j = lenA, lenB
    alnseqA, alnseqB = list(), list()
    edit_distance = 0
    ops = []

    while (i > 0 and j > 0):
        if traceback_matrix[i][j] == 0:
            alnseqA.append(seqA[i-1])
            alnseqB.append(seqB[j-1])
            ops.append((seqA[i-1], seqB[j-1]))
            if seqA[i-1] != seqB[j-1]:
                edit_distance += 1
            i -= 1
            j -= 1
        elif traceback_matrix[i][j] == 1:
            alnseqA.append(seqA[i-1])
            alnseqB.append('*')
            ops.append((seqA[i-1], "*"))
            edit_distance += 1
            i -= 1
        elif traceback_matrix[i][j] == -1:
            alnseqA.append('*')
            alnseqB.append(seqB[j-1])

            ops.append(("*", seqB[j-1]))

            edit_distance += 1
            j -= 1
    while i > 0:
        alnseqA.append(seqA[i-1])
        alnseqB.append('*')

        ops.append((seqA[i-1], "*"))

        edit_distance += 1
        i -= 1
    while j > 0:
        alnseqA.append('*')
        alnseqB.append(seqB[j-1])

        ops.append(("*", seqB[j-1]))


        edit_distance += 1
        j -= 1

    # reverse sequences to original orders
    alnseqA = alnseqA[::-1]
    alnseqB = alnseqB[::-1]
    ops = ops[::-1]

    # print(alnseqA)
    # print(alnseqB)
    # print('edit distance: ', edit_distance)
    return [alnseqA, alnseqB, edit_distance, ops]


_MAX_EXAMPLES = 5

def collect_errors(ref, hyp):
    """Collect substitution, deletion, and insertion statistics across all word pairs.

    Returns a dict with:
        n_sub, n_del, n_ins          – total counts per error type
        sub_ref_counter              – Counter of ref phones involved in substitutions
        sub_hyp_counter              – Counter of hyp phones involved in substitutions
        del_counter                  – Counter of deleted ref phones
        ins_counter                  – Counter of inserted hyp phones
        sub_pair_counter             – Counter of (ref_phone, hyp_phone) substitution pairs
        sub_ref_examples             – {phone: [example dicts]} up to _MAX_EXAMPLES each
        sub_hyp_examples             – {phone: [example dicts]} up to _MAX_EXAMPLES each
        del_examples                 – {phone: [example dicts]} up to _MAX_EXAMPLES each
        ins_examples                 – {phone: [example dicts]} up to _MAX_EXAMPLES each
        sub_pair_examples            – {(r,h): [example dicts]} up to _MAX_EXAMPLES each

    Each example dict: {'word': str, 'ref': str, 'hyp': str, 'op': str}
    Only words present in both ref and hyp are evaluated.
    """
    n_sub = n_del = n_ins = 0
    sub_ref_counter  = Counter()
    sub_hyp_counter  = Counter()
    del_counter      = Counter()
    ins_counter      = Counter()
    sub_pair_counter = Counter()

    sub_ref_examples  = defaultdict(list)
    sub_hyp_examples  = defaultdict(list)
    del_examples      = defaultdict(list)
    ins_examples      = defaultdict(list)
    sub_pair_examples = defaultdict(list)

    for word, ref_phones in ref.items():
        if word not in hyp:
            continue
        hyp_phones = hyp[word]
        _, _, _, ops = align_trxn(ref_phones, hyp_phones)

        base = {
            'word': word,
            'ref':  ' '.join(ref_phones),
            'hyp':  ' '.join(hyp_phones),
        }

        for (r, h) in ops:
            if r == '*':
                n_ins += 1
                ins_counter[h] += 1
                if len(ins_examples[h]) < _MAX_EXAMPLES:
                    ins_examples[h].append({**base, 'op': f'ins({h})'})
            elif h == '*':
                n_del += 1
                del_counter[r] += 1
                if len(del_examples[r]) < _MAX_EXAMPLES:
                    del_examples[r].append({**base, 'op': f'del({r})'})
            elif r != h:
                n_sub += 1
                sub_ref_counter[r] += 1
                sub_hyp_counter[h] += 1
                sub_pair_counter[(r, h)] += 1
                if len(sub_ref_examples[r]) < _MAX_EXAMPLES:
                    sub_ref_examples[r].append({**base, 'op': f'sub({r}→{h})'})
                if len(sub_hyp_examples[h]) < _MAX_EXAMPLES:
                    sub_hyp_examples[h].append({**base, 'op': f'sub({r}→{h})'})
                pair = (r, h)
                if len(sub_pair_examples[pair]) < _MAX_EXAMPLES:
                    sub_pair_examples[pair].append({**base, 'op': f'sub({r}→{h})'})

    return {
        'n_sub': n_sub,
        'n_del': n_del,
        'n_ins': n_ins,
        'sub_ref_counter':  sub_ref_counter,
        'sub_hyp_counter':  sub_hyp_counter,
        'del_counter':      del_counter,
        'ins_counter':      ins_counter,
        'sub_pair_counter': sub_pair_counter,
        'sub_ref_examples':  dict(sub_ref_examples),
        'sub_hyp_examples':  dict(sub_hyp_examples),
        'del_examples':      dict(del_examples),
        'ins_examples':      dict(ins_examples),
        'sub_pair_examples': dict(sub_pair_examples),
    }

err = collect_errors(refs_dict, hyps_dict)
total_errors = err["n_sub"] + err["n_del"] + err["n_ins"]

error_div = pd.DataFrame([
    {"Type": "Substitution", "Count": err["n_sub"], "% of Errors": f"{err['n_sub'] / total_errors * 100:.1f}%"},
    {"Type": "Deletion",     "Count": err["n_del"], "% of Errors": f"{err['n_del'] / total_errors * 100:.1f}%"},
    {"Type": "Insertion",    "Count": err["n_ins"], "% of Errors": f"{err['n_ins'] / total_errors * 100:.1f}%"},
    {"Type": "TOTAL",        "Count": total_errors, "% of Errors": "100.0%"},
])
print(f"Total edit operations: {total_errors}")
error_div

Total edit operations: 324


,Type,Count,% of Errors
0,Substitution,33,10.2%
1,Deletion,152,46.9%
2,Insertion,139,42.9%
3,TOTAL,324,100.0%


As seen above, most of the mistakes made by the model involved deletion and insertion, while only 10.2% of the errors were from substitution.

In [11]:
def top10(counter, label):
    print(f"--- {label} ---")
    return pd.DataFrame([{"Phone": p, "Count": c} for p, c in counter.most_common(10)])

print("Ref phones most often substituted:")
display(top10(err["sub_ref_counter"], "REF phones replaced"))
print("Ref phones most often deleted:")
display(top10(err["del_counter"], "REF phones dropped"))
print("Phones most often inserted:")
display(top10(err["ins_counter"], "phones added"))

Ref phones most often substituted:
--- REF phones replaced ---


,Phone,Count
0,j,10
1,w,5
2,b,4
3,v,2
4,n,2
5,ŋ,2
6,a,2
7,h,2
8,i,1
9,t͡ʃ,1


Ref phones most often deleted:
--- REF phones dropped ---


,Phone,Count
0,ʔ,128
1,a,5
2,k,4
3,s,3
4,b,3
5,ɡ,2
6,w,1
7,d͡ʒ,1
8,o,1
9,p,1


Phones most often inserted:
--- phones added ---


,Phone,Count
0,ʔ,134
1,u,2
2,o,1
3,t,1
4,i,1


In [12]:
pair_rows = [
    {"Pair": f"{r} -> {h}", "Count": c}
    for (r, h), c in err["sub_pair_counter"].most_common(10)
]
display(pd.DataFrame(pair_rows))

print("\nExample words for the top substitution pairs (critical phonetic shifts):")
for (r, h), c in err["sub_pair_counter"].most_common(5):
    print(f"\n  {r} -> {h}  (count={c})")
    for ex in err["sub_pair_examples"].get((r, h), [])[:3]:
        print(f"    {ex['word']:<20}  ref:[{ex['ref']}]  hyp:[{ex['hyp']}]")

,Pair,Count
0,j -> i,8
1,b -> v,4
2,v -> b,2
3,w -> u,2
4,w -> ʔ,2
5,n -> ʔ,1
6,ŋ -> ɡ,1
7,w -> o,1
8,i -> ɡ,1
9,t͡ʃ -> s,1



Example words for the top substitution pairs (critical phonetic shifts):

  j -> i  (count=8)
    1447                  ref:[b e j n t e]  hyp:[b e ʔ i n t e]
    1880                  ref:[b o j n a]  hyp:[b o ʔ i n a]
    2190                  ref:[k o l o n j a]  hyp:[k o l o n i a]

  b -> v  (count=4)
    65                    ref:[b i]  hyp:[v i]
    1954                  ref:[ʔ a b e ɾ e j d s]  hyp:[ʔ a v e ɾ e j d s]
    2396                  ref:[ʔ o b e ɾ]  hyp:[ʔ o v e ɾ]

  v -> b  (count=2)
    27                    ref:[v e l o s i d a d]  hyp:[b e l o s i d a d]
    3183                  ref:[l a v e ɾ n]  hyp:[l a b e ɾ n]

  w -> u  (count=2)
    353                   ref:[t e s a w ɾ o]  hyp:[t e s a ʔ u ɾ o]
    2146                  ref:[ʔ a w d j o]  hyp:[ʔ a ʔ u d j o]

  w -> ʔ  (count=2)
    817                   ref:[b i l a w o]  hyp:[b i l a ʔ u ʔ o]
    1293                  ref:[m a m i t w i n]  hyp:[m a m i t u ʔ i n]


In [13]:
print("Example words for the top deleted ref phones (dropped in hyp):")
for r, c in err["del_counter"].most_common(3):
    print(f"\n  del({r})  (count={c})")
    for ex in err["del_examples"].get(r, [])[:3]:
        print(f"    {ex['word']:<20}  ref:[{ex['ref']}]  hyp:[{ex['hyp']}]")

print("\nExample words for the top inserted phones (added in hyp):")
for h, c in err["ins_counter"].most_common(3):
    print(f"\n  ins({h})  (count={c})")
    for ex in err["ins_examples"].get(h, [])[:3]:
        print(f"    {ex['word']:<20}  ref:[{ex['ref']}]  hyp:[{ex['hyp']}]")


print("\nExample words for the top substituted phones (substituted in hyp):")
for h, c in err["sub_ref_counter"].most_common(3):
    print(f"\n  sub({h})  (count={c})")
    for ex in err["sub_ref_examples"].get(h, [])[:3]:
        print(f"    {ex['word']:<20}  ref:[{ex['ref']}]  hyp:[{ex['hyp']}]")

Example words for the top deleted ref phones (dropped in hyp):

  del(ʔ)  (count=128)
    10                    ref:[m a ŋ i s d a ʔ]  hyp:[m a ŋ ʔ i s d a]
    29                    ref:[k a p i t b a n s a ʔ]  hyp:[k a p i t b a n s a]
    34                    ref:[k a b u t ʔ o]  hyp:[k a b u t o]

  del(a)  (count=5)
    1339                  ref:[p i n a k a n a k a p a ɡ p a p a b a ɡ a b a ɡ]  hyp:[p i n a k a n a k a p a ɡ p a ɡ]
    1339                  ref:[p i n a k a n a k a p a ɡ p a p a b a ɡ a b a ɡ]  hyp:[p i n a k a n a k a p a ɡ p a ɡ]
    1339                  ref:[p i n a k a n a k a p a ɡ p a p a b a ɡ a b a ɡ]  hyp:[p i n a k a n a k a p a ɡ p a ɡ]

  del(k)  (count=4)
    1014                  ref:[b u l a k k a h o j]  hyp:[b u l a k a h o j]
    1633                  ref:[t a t a k k a l a k a l]  hyp:[t a t a k a l a k a l]
    2510                  ref:[p a ɡ k a m a k a b a ɡ u n s a k u p i n]  hyp:[p a ɡ k a m a k a b a ɡ u n s i p a]

Example words for 

### Glottal stop errors
The most common instance of phoneme error made by the model is from the insertion or deletion of the glottal stop (ʔ). The code below demonstrates the location of the glottal stop errors in the model's predictions, whether they are **initial, medial or final** in position. For insertions, the glottal stop errors are mostly word-final, while for deletions, the distribution is nearly even between word-medial and word-final errors. 

The word-final errors are a result of Filipino spelling's lack of indication regarding word-final `ʔ`. This makes the use of glottal stops significantly more ambigious to the model, resulting in the deletion or insertion of word-final glottal stops where they are not needed. The word-medial errors may result from a similar ambiguity in the data, where words such maya-maya or pinag-iinitan are not necessarily spelled with **-** in the middle that would otherwise be a potential indicator of word-medial `ʔ`.

In [14]:
GLOTTAL = "ʔ"
_MAX_POS_EXAMPLES = 8


def _pos_bucket(idx, n):
    """Classify a phone index within a word of length n."""
    if idx == 0:
        return "initial"
    if idx == n - 1:
        return "final"
    return "medial"


In [15]:
del_pos = Counter()
ins_pos = Counter()
del_medial_examples = []
del_final_examples = []   # (word, ref_str, hyp_str) — real final ʔ dropped
ins_final_examples = []   # (word, ref_str, hyp_str) — bogus final ʔ added

for word, ref_phones in refs_dict.items():
    if word not in hyps_dict:
        continue
    hyp_phones = hyps_dict[word]
    alnA, alnB, _, _ = align_trxn(ref_phones, hyp_phones)
    ref_str, hyp_str = " ".join(ref_phones), " ".join(hyp_phones)

    ri = hi = 0
    for a, b in zip(alnA, alnB):
        if a == GLOTTAL and b == "*":          # deletion: ref has ʔ, hyp dropped it
            pos = _pos_bucket(ri, len(ref_phones))
            del_pos[pos] += 1
            if pos == "medial" and len(del_medial_examples) < _MAX_POS_EXAMPLES:
                del_medial_examples.append((word, ref_str, hyp_str))
            if pos == "final" and len(del_final_examples) < _MAX_POS_EXAMPLES:
                del_final_examples.append((word, ref_str, hyp_str))
        if b == GLOTTAL and a == "*":          # insertion: hyp added a spurious ʔ
            pos = _pos_bucket(hi, len(hyp_phones))
            ins_pos[pos] += 1
            if pos == "final" and len(ins_final_examples) < _MAX_POS_EXAMPLES:
                ins_final_examples.append((word, ref_str, hyp_str))
        if a != "*":
            ri += 1
        if b != "*":
            hi += 1


def _pos_table(counter, label):
    total = sum(counter.values())
    rows = []
    for pos in ("initial", "medial", "final"):
        c = counter[pos]
        pct = f"{c / total * 100:.1f}%" if total else "-"
        rows.append({"Position": pos, "Count": c, "% of ʔ errors": pct})
    rows.append({"Position": "TOTAL", "Count": total, "% of ʔ errors": "100.0%"})
    print(f"--- {label} ---")
    return pd.DataFrame(rows)


print("Where in the word is the glottal stop ʔ mispredicted?\n")
display(_pos_table(del_pos, "ʔ DELETIONS  (ref has ʔ, model dropped it)"))
display(_pos_table(ins_pos, "ʔ INSERTIONS (model added a spurious ʔ)"))

# Concrete word-final examples: every one is a vowel-final spelling, so the same
# surface cue maps to opposite correct answers — unsolvable from orthography alone.
print("\nWord-medial ʔ DELETIONS (a real medial ʔ was dropped):")
for w, r, h in del_medial_examples:
    print(f"    {w:16}  ref:[{r}]   hyp:[{h}]")


print("\nWord-final ʔ DELETIONS (a real final ʔ was dropped):")
for w, r, h in del_final_examples:
    print(f"    {w:16}  ref:[{r}]   hyp:[{h}]")

print("\nWord-final ʔ INSERTIONS (a bogus final ʔ was added):")
for w, r, h in ins_final_examples:
    print(f"    {w:16}  ref:[{r}]   hyp:[{h}]")

Where in the word is the glottal stop ʔ mispredicted?

--- ʔ DELETIONS  (ref has ʔ, model dropped it) ---


,Position,Count,% of ʔ errors
0,initial,0,0.0%
1,medial,60,46.9%
2,final,68,53.1%
3,TOTAL,128,100.0%


--- ʔ INSERTIONS (model added a spurious ʔ) ---


,Position,Count,% of ʔ errors
0,initial,0,0.0%
1,medial,36,26.9%
2,final,98,73.1%
3,TOTAL,134,100.0%



Word-medial ʔ DELETIONS (a real medial ʔ was dropped):
                  34  ref:[k a b u t ʔ o]   hyp:[k a b u t o]
                 107  ref:[t a k a w ʔ a l a t]   hyp:[t a k a w a l a t]
                 128  ref:[d a ɡ ʔ i s]   hyp:[d a ɡ i s]
                 142  ref:[m a t a b ʔ a ŋ]   hyp:[m a t a b a ŋ]
                 288  ref:[m a j a ʔ m a j a ʔ]   hyp:[m a j a m a j a]
                 334  ref:[p i n a ɡ ʔ i ʔ i n i t a n]   hyp:[p i n a ɡ i ʔ i n i t a n]
                 342  ref:[t a m ʔ i s h a ŋ h a ŋ]   hyp:[t a m i s h a ŋ h a ŋ]
                 351  ref:[t a n ʔ a w]   hyp:[t a n a w]

Word-final ʔ DELETIONS (a real final ʔ was dropped):
                  10  ref:[m a ŋ i s d a ʔ]   hyp:[m a ŋ ʔ i s d a]
                  29  ref:[k a p i t b a n s a ʔ]   hyp:[k a p i t b a n s a]
                  39  ref:[b a l i k t a j a ʔ]   hyp:[b a l i k t a j a]
                  43  ref:[t a ŋ ɡ i ŋ ɡ i ʔ]   hyp:[t a ŋ ɡ i ŋ ɡ i]
                  98  ref:[m a ɡ k a s

### Vowel Confusion

Another potential source of error stems from e↔i or o↔u substitution, as both pairs are allophonic in spoken Filipino, meaning they can act as spelling variations of the same sound. However, as seen below, the model is able to distinguish between the two completely.

In [16]:
import unicodedata
_DEFAULT_VOWEL_GROUPS = {"o/u": {"o", "u"}, "e/i": {"e", "i"}}
_LENGTH   = 'ː'  


def _base_vowel(ph):
    """Strip combining marks and the length modifier (ː) so oː/õ fold to o."""
    return ''.join(
        c for c in ph
        if not unicodedata.category(c).startswith('M') and c != _LENGTH
    )


def vowel_confusion(ref, hyp, groups=None):
    """Count directed vowel-height substitutions per group (e.g. o↔u, e↔i).

    Reuses collect_errors(); a substitution (r→h) counts for a group when the
    base vowels of r and h are both in that group and differ. Length/diacritic
    variants (oː, õ) fold to their base vowel via _base_vowel.

    Args:
        ref, hyp: dicts of {word: [phones]} (only words in both are scored).
        groups:   {label: set of base vowels}; defaults to o/u and e/i.

    Returns {label: {"count": int,
                     "by_pair": Counter{(ref_phone, hyp_phone): n},
                     "examples": [example dicts],
                     "rate": float}}   where rate = count / total substitutions.
    """
    if groups is None:
        groups = _DEFAULT_VOWEL_GROUPS

    err = collect_errors(ref, hyp)
    n_sub = err['n_sub']

    result = {}
    for label, vowels in groups.items():
        by_pair = Counter()
        examples = []
        for (r, h), c in err['sub_pair_counter'].items():
            rb, hb = _base_vowel(r), _base_vowel(h)
            if rb in vowels and hb in vowels and rb != hb:
                by_pair[(r, h)] += c
                examples.extend(err['sub_pair_examples'].get((r, h), []))
        count = sum(by_pair.values())
        result[label] = {
            'count':    count,
            'by_pair':  by_pair,
            'examples': examples,
            'rate':     count / n_sub if n_sub else 0.0,
        }
    return result

In [17]:
vc = vowel_confusion(refs_dict, hyps_dict)   # default groups: o/u and e/i


display(pd.DataFrame([
    {"Confusion": k, "Count": v["count"], "% of subs": f"{v['rate'] * 100:.1f}%"}
    for k, v in vc.items()
]))

for label, v in vc.items():
    print(f"\n{label}: {v['count']} confusions")
    for (r, h), c in v["by_pair"].most_common():
        print(f"    {r} -> {h}: {c}")
    for ex in v["examples"][:5]:
        print(f"      {ex['word']:<18} ref:[{ex['ref']}]  hyp:[{ex['hyp']}]")

if all(v["count"] == 0 for v in vc.values()):
    print("\nNo o↔u or e↔i substitutions: the phonemic model keeps these vowels distinct.")

,Confusion,Count,% of subs
0,o/u,0,0.0%
1,e/i,0,0.0%



o/u: 0 confusions

e/i: 0 confusions

No o↔u or e↔i substitutions: the phonemic model keeps these vowels distinct.


### Code switching

While out of scope, the performance was tested on a 20-word dataset consisting of mixed Tagalog, English, and Taglish words.

In [11]:
import glob

# 200k model cheptions
ensemble_paths = sorted(
    glob.glob("models/5090Attempt/200kattempt/**/*.pt", recursive=True)
)

print(f"\nEnsemble of {len(ensemble_paths)} Tagalog checkpoints:")
for p in ensemble_paths:
    print(f"  - {p}")


Ensemble of 12 Tagalog checkpoints:
  - models/5090Attempt/200kattempt\seed_42\ckpt_seed42_checkpoint100000.pt
  - models/5090Attempt/200kattempt\seed_42\ckpt_seed42_checkpoint150000.pt
  - models/5090Attempt/200kattempt\seed_42\ckpt_seed42_checkpoint200000.pt
  - models/5090Attempt/200kattempt\seed_42\ckpt_seed42_checkpoint50000.pt
  - models/5090Attempt/200kattempt\seed_67\ckpt_seed67_checkpoint100000.pt
  - models/5090Attempt/200kattempt\seed_67\ckpt_seed67_checkpoint150000.pt
  - models/5090Attempt/200kattempt\seed_67\ckpt_seed67_checkpoint200000.pt
  - models/5090Attempt/200kattempt\seed_67\ckpt_seed67_checkpoint50000.pt
  - models/5090Attempt/200kattempt\seed_888\ckpt_seed888_checkpoint100000.pt
  - models/5090Attempt/200kattempt\seed_888\ckpt_seed888_checkpoint150000.pt
  - models/5090Attempt/200kattempt\seed_888\ckpt_seed888_checkpoint200000.pt
  - models/5090Attempt/200kattempt\seed_888\ckpt_seed888_checkpoint50000.pt


In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cs_per, cs_wer, cs_refs, cs_hyps = batch_evaluate_checkpoint_beam(
    model_paths=ensemble_paths,
    src_path="data/splits/tgl_codeswitch/test_codeswitch.src",
    tgt_path="data/splits/tgl_codeswitch/test_codeswitch.tgt",
    device=device,
    batch_size=64,
    beam=5,
)

print(
    f"\nCode-Switching Evaluation ({len(ensemble_paths)} models): PER={cs_per:.4f}  WER={cs_wer:.4f}"
)

  decoded 20/20

Code-Switching Evaluation (12 models): PER=0.2411  WER=0.6500


It managed to get a PER of 24.11%, and a WER of 65.00%. These results indicate that the model does not really perform well with code-switching.

In [21]:
with open("data/splits/tgl_codeswitch/test_codeswitch.src", encoding="utf-8") as f:
    src_words = [line.strip() for line in f if line.strip()]

df_cs = pd.DataFrame({
    "input_word": src_words,
    "target_ref": cs_refs,
    "model_hyp": cs_hyps
})

df_cs["is_error"] = df_cs["target_ref"] != df_cs["model_hyp"]

errors_only = df_cs[df_cs["is_error"] == True]

refs_list = [str(x).strip().split() for x in df_cs["target_ref"]]
hyps_list = [str(x).strip().split() for x in df_cs["model_hyp"]]
refs_dict = dict(enumerate(refs_list))
hyps_dict = dict(enumerate(hyps_list))

print(errors_only.sample(10))

                           input_word                      target_ref  \
6       <tagalog> i - d o w n l o a d  [ʔ, i, d, a, w, n, l, o, w, d]   
12                <tagalog> x - r a y           [ʔ, e, k, s, ɾ, e, j]   
19              <tagalog> s o c i a l              [s, o, s, j, a, l]   
1           <tagalog> f a c e b o o k           [f, e, j, s, b, u, k]   
13          <tagalog> c o m p u t e r     [k, o, m, p, j, u, t, e, ɾ]   
7         <tagalog> m a g - o r d e r     [m, a, ɡ, ʔ, o, ɾ, d, e, ɾ]   
5   <tagalog> m a g - c h e c k - i n   [m, a, ɡ, t͡ʃ, e, k, ʔ, i, n]   
11            <tagalog> v a c c i n e              [b, a, k, s, i, n]   
8     <tagalog> n a k a - l o g - i n  [n, a, k, a, l, o, ɡ, ʔ, i, n]   
18        <tagalog> f a s t - f o o d           [f, a, s, t, f, u, d]   

                                  model_hyp  is_error  
6   [ʔ, i, ʔ, i, d, o, w, n, l, o, ʔ, a, d]      True  
12                          [s, i, ɾ, a, j]      True  
19                       [s,

### Sentence Prediction

The vocabulary does not include spaces `" "`, meaning that phrases or lines with spaces cannot be properly represented.

### Stress Prediction

In Tagalog, stress is important as similarly-spelled words can have entirely different meanings based on how they are pronounced. In everyday text, stress is not included and the meaning is inferred based on surrounding context.

In [13]:
import glob
import torch
import tempfile
import os

# Load
ensemble_paths = []
seeds = [42, 67, 888]
for seed in seeds:
    ensemble_paths.extend(
        sorted(glob.glob(f"models/5090Attempt/200kattempt/seed_{seed}/ckpt_*.pt"))
    )

print(f"Loaded {len(ensemble_paths)} checkpoints for ensemble evaluation.\n")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Test cases
test_cases = [
    # --- Stress Prediction & Lack of Information (Homographs) ---
    {
        "word": "buhay",
        "category": "Stress / Information Deficit",
        "meanings": "/ˈbuhaj/ (life) vs /buˈhaj/ (alive)",
    },
    {
        "word": "paso",
        "category": "Stress / Information Deficit",
        "meanings": "/paˈsoʔ/ (flowerpot) vs /ˈpasoʔ/ (expired)",
    },
    {
        "word": "pito",
        "category": "Stress / Information Deficit",
        "meanings": "/ˈpito/ (whistle) vs /piˈto/ (seven)",
    },
    # --- Morphology Recovery Requires Context ---
    {
        "word": "basa",
        "category": "Morphology & Context",
        "meanings": "Isolated root ('read' vs 'wet') without sentence syntax",
    },
    {
        "word": "tanong",
        "category": "Morphology & Context",
        "meanings": "Isolated root ('ask') without enclitic/syntactic context",
    },
]

# Define the wrapper around batch_evaluate_checkpoint_beam
def predict_g2p_ensemble(words, ensemble_paths, device, beam=5, batch_size=64):
    formatted_inputs = [f"<tagalog> {' '.join(list(w))}" for w in words]

    with tempfile.NamedTemporaryFile("w+", delete=False, encoding="utf-8") as f_src, \
         tempfile.NamedTemporaryFile("w+", delete=False, encoding="utf-8") as f_tgt:

        f_src.write("\n".join(formatted_inputs) + "\n")
        f_tgt.write("\n".join(["dummy"] * len(words)) + "\n")

        src_path = f_src.name
        tgt_path = f_tgt.name

    try:
        _, _, _, predictions = batch_evaluate_checkpoint_beam(
            model_paths=ensemble_paths,
            src_path=src_path,
            tgt_path=tgt_path,
            device=device,
            batch_size=batch_size,
            beam=beam,
        )
    finally:
        os.remove(src_path)
        os.remove(tgt_path)

    return predictions

# Run predictions for words
target_words = [case["word"] for case in test_cases]
predictions = predict_g2p_ensemble(target_words, ensemble_paths, device=device, beam=5)

# Output
print("=" * 70)
print(f"{'WORD':<10} | {'CATEGORY':<28} | {'MODEL PHONEMIC OUTPUT'}")
print("=" * 70)

for case, output_phonemes in zip(test_cases, predictions):
    word = case["word"]
    category = case["category"]

    print(f"{word:<10} | {category:<28} | {output_phonemes}")
    print(f"   └─ Target Ambiguity: {case['meanings']}")
    print("-" * 70)

Loaded 12 checkpoints for ensemble evaluation.

  decoded 5/5
WORD       | CATEGORY                     | MODEL PHONEMIC OUTPUT
buhay      | Stress / Information Deficit | ['b', 'u', 'h', 'a', 'j']
   └─ Target Ambiguity: /ˈbuhaj/ (life) vs /buˈhaj/ (alive)
----------------------------------------------------------------------
paso       | Stress / Information Deficit | ['p', 'a', 's', 'o', 'ʔ']
   └─ Target Ambiguity: /paˈsoʔ/ (flowerpot) vs /ˈpasoʔ/ (expired)
----------------------------------------------------------------------
pito       | Stress / Information Deficit | ['p', 'i', 't', 'o', 'ʔ']
   └─ Target Ambiguity: /ˈpito/ (whistle) vs /piˈto/ (seven)
----------------------------------------------------------------------
basa       | Morphology & Context         | ['b', 'a', 's', 'a', 'ʔ']
   └─ Target Ambiguity: Isolated root ('read' vs 'wet') without sentence syntax
----------------------------------------------------------------------
tanong     | Morphology & Context       

Because predictions for single words lack surrounding sentence context, these words end up with only a single pronunciation of it. `buhay` is only predicted as `['b', 'u', 'h', 'a', 'j']` without any stress markers.

### Long Input

The maximum input and output lengths will be tested here.

In [30]:
# Base word
base_word = "pinagkakatiwalaan"  # 17 chars

# Test inputs length
test_lengths = [10, 20, 35, 50, 75, 100, 125, 150, 200, 225, 250]
length_test_cases = []

for target_len in test_lengths:
    # Repeat base_word until reach or exceed target_len
    repeated = (base_word * ((target_len // len(base_word)) + 1))[:target_len]
    length_test_cases.append({
        "input_str": repeated,
        "length": len(repeated)
    })

print("Running length stress test across varying input sizes...\n")

inputs = [case["input_str"] for case in length_test_cases]

try:
    predictions = predict_g2p_ensemble(inputs, ensemble_paths, device=device, beam=5)

    print("=" * 75)
    print(f"{'INPUT LENGTH':<12} | {'INPUT PREVIEW':<25} | {'OUTPUT PHONEME COUNT'}")
    print("=" * 75)

    for case, output_phonemes in zip(length_test_cases, predictions):
        inp = case["input_str"]
        inp_len = case["length"]
        out_len = len(output_phonemes)
        preview = inp if len(inp) <= 22 else inp[:20] + "..."

        print(f"{inp_len:<12} | {preview:<25} | {out_len} tokens")
        # Print part of output to see if it cuts off or gets stuck in loop
        out_snippet = "".join(output_phonemes)
        if len(out_snippet) > 30:
            out_snippet = out_snippet[:27] + "..."
        print(f"   └─ Output snippet: {out_snippet}")
        print("-" * 75)

except Exception as e:
    print(f"\n[!] Model crashed during length testing: {e}")

Running length stress test across varying input sizes...

  decoded 11/11
INPUT LENGTH | INPUT PREVIEW             | OUTPUT PHONEME COUNT
10           | pinagkakat                | 10 tokens
   └─ Output snippet: pinaɡkakat
---------------------------------------------------------------------------
20           | pinagkakatiwalaanpin      | 18 tokens
   └─ Output snippet: pinaɡkakatiwalaʔan
---------------------------------------------------------------------------
35           | pinagkakatiwalaanpin...   | 18 tokens
   └─ Output snippet: pinaɡkakatiwalikan
---------------------------------------------------------------------------
50           | pinagkakatiwalaanpin...   | 18 tokens
   └─ Output snippet: pinaɡkakatiwalikan
---------------------------------------------------------------------------
75           | pinagkakatiwalaanpin...   | 18 tokens
   └─ Output snippet: pinaɡkakatiwalikan
---------------------------------------------------------------------------
100          | pinag

Testing input lengths from 10 to 250 characters shows that there is a length limitation. At longer character lengths, the model usually predicts the `<EOS>` token after 18 phonemes. Very long input lengths beyond 100 causes even earlier termination, dropping the output down to 14 phonemes.

### Per-model Evaluation

In this section, we will test out how each model performs on their own, without ensembling. By running individual beam search decoding (beam = 5) across all 12 checkpoints, we can measure how much of an improvement ensembling actually provides.

In [14]:
# Input words
src_path = "data/splits/tgl_test/test.src"
with open(src_path, "r", encoding="utf-8") as f:
    input_words = [line.strip() for line in f]

# Store individual results
individual_results = []

# Loop through each model
for path in ensemble_paths:
    per, wer, refs, hyps = batch_evaluate_checkpoint_beam(
        model_paths=[path],
        src_path="data/splits/tgl_test/test.src",
        tgt_path="data/splits/tgl_test/test.tgt",
        device=device,
        batch_size=64,
        beam=5,
    )

    filename = Path(path).name
    stem_name = Path(path).stem

    # Extract seed and checkpoint number from the file
    individual_results.append(
        {"model_path": path, "filename": filename, "PER": per, "WER": wer}
    )

    # CSV columns
    df_samples = pd.DataFrame(
        {
            "input_word": input_words,
            "target_ref": refs,
            "model_hyp": hyps,
            "is_error": [
                str(r).strip() != str(h).strip() for r, h in zip(refs, hyps)
            ],
        }
    )
    df_samples.to_csv(f"{stem_name}_test_results.csv", index=False)

# Convert results to DataFrame
df_individual = pd.DataFrame(individual_results)
df_individual = df_individual.sort_values(by="PER").reset_index(drop=True)

display(df_individual)

  decoded 64/3259
  decoded 128/3259
  decoded 192/3259
  decoded 256/3259
  decoded 320/3259
  decoded 384/3259
  decoded 448/3259
  decoded 512/3259
  decoded 576/3259
  decoded 640/3259
  decoded 704/3259
  decoded 768/3259
  decoded 832/3259
  decoded 896/3259
  decoded 960/3259
  decoded 1024/3259
  decoded 1088/3259
  decoded 1152/3259
  decoded 1216/3259
  decoded 1280/3259
  decoded 1344/3259
  decoded 1408/3259
  decoded 1472/3259
  decoded 1536/3259
  decoded 1600/3259
  decoded 1664/3259
  decoded 1728/3259
  decoded 1792/3259
  decoded 1856/3259
  decoded 1920/3259
  decoded 1984/3259
  decoded 2048/3259
  decoded 2112/3259
  decoded 2176/3259
  decoded 2240/3259
  decoded 2304/3259
  decoded 2368/3259
  decoded 2432/3259
  decoded 2496/3259
  decoded 2560/3259
  decoded 2624/3259
  decoded 2688/3259
  decoded 2752/3259
  decoded 2816/3259
  decoded 2880/3259
  decoded 2944/3259
  decoded 3008/3259
  decoded 3072/3259
  decoded 3136/3259
  decoded 3200/3259
  decoded 3259/3

,model_path,filename,PER,WER
0,models/5090Attempt/200kattempt/seed_888\ckpt_s...,ckpt_seed888_checkpoint150000.pt,0.014109,0.097269
1,models/5090Attempt/200kattempt/seed_42\ckpt_se...,ckpt_seed42_checkpoint100000.pt,0.014425,0.097883
2,models/5090Attempt/200kattempt/seed_888\ckpt_s...,ckpt_seed888_checkpoint100000.pt,0.014740,0.099724
3,models/5090Attempt/200kattempt/seed_42\ckpt_se...,ckpt_seed42_checkpoint200000.pt,0.016159,0.095735
4,models/5090Attempt/200kattempt/seed_67\ckpt_se...,ckpt_seed67_checkpoint200000.pt,0.016356,0.095428
5,models/5090Attempt/200kattempt/seed_888\ckpt_s...,ckpt_seed888_checkpoint200000.pt,0.016356,0.097576
6,models/5090Attempt/200kattempt/seed_42\ckpt_se...,ckpt_seed42_checkpoint150000.pt,0.016474,0.096962
7,models/5090Attempt/200kattempt/seed_67\ckpt_se...,ckpt_seed67_checkpoint150000.pt,0.017105,0.099110
8,models/5090Attempt/200kattempt/seed_42\ckpt_se...,ckpt_seed42_checkpoint50000.pt,0.017184,0.108315
9,models/5090Attempt/200kattempt/seed_67\ckpt_se...,ckpt_seed67_checkpoint50000.pt,0.017617,0.101872


We can see that they all perform worse compared to the ensemble, with the best PER being 0.0141 (1.41%) by seed888_checkpoint150000 and WER being 0.0954 (9.54%) by seed67_checkpoint200000. Compared to our ensemble's result with a PER of 0.0128 (1.28%) and a WER of 0.0871 (8.71%) This confirms that ensembling does provide an improvement in results.

### Comparison to reference work
Comparison to other Filipino G2Ps from referenced work, they are not direct comparisons due to beign tested on different data sets but offer noteworthy insight regardless.

|System | PER | Dataset|
|-----|-----|------|
| This model (Salo-Salo) | 1.28% | Wikitionary test split from this work|
|Aquino et al. (2019) - monolingual statistical	model |5.87%	 |Transcribed speaker corpus|
|Deri & Knight (2016) - cross-lingual WFST	|5-10%	|Cross-lingual Wiktionary|